# FFAI Structured Output

This notebook demonstrates FFAI's structured output feature, which validates
LLM responses against **Pydantic models** with automatic retry on validation
failure.

New features covered:

1. **`response_model=` parameter** -- pass a Pydantic model to `generate_response()`
2. **Automatic schema injection** -- JSON Schema appended to system instructions
3. **Validation retry loop** -- on failure, the LLM gets error feedback and retries
4. **`ResponseResult.parsed`** -- the validated Pydantic model instance

This notebook uses the **real Mistral API** via `FFMistralSmall`.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from pydantic import BaseModel, Field

from src.Clients.FFMistralSmall import FFMistralSmall
from src.FFAI import FFAI

client = FFMistralSmall(
    max_tokens=1024,
    temperature=0.3,
)

ffai = FFAI(client)

print(f"FFAI initialized with model: {client.model}")

---

## Step 1: Define a Pydantic Model

Any Pydantic `BaseModel` subclass works. The model's JSON Schema is
automatically extracted and injected into the system prompt.

In [ ]:
class SentimentAnalysis(BaseModel):
    label: str = Field(description="One of: positive, negative, neutral")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    key_phrases: list[str] = Field(description="Important phrases from the text")

print(SentimentAnalysis.model_json_schema())

---

## Step 2: Single Structured Output Call

Pass `response_model=SentimentAnalysis` to `generate_response()`. FFAI:

1. Converts the model to JSON Schema
2. Appends schema instructions to the system prompt
3. Calls the LLM
4. Validates the response with Pydantic
5. Returns `result.parsed` as a validated model instance

In [ ]:
result = ffai.generate_response(
    prompt="Analyze the sentiment of this customer review: 'The product quality exceeded my expectations. Shipping was fast and the packaging was excellent. Will definitely order again.'",
    prompt_name="sentiment_1",
    response_model=SentimentAnalysis,
)

print(f"Status:     {result.status}")
print(f"Raw JSON:   {result.response}")
print()
if result.parsed:
    print(f"Label:      {result.parsed.label}")
    print(f"Confidence: {result.parsed.confidence}")
    print(f"Phrases:    {result.parsed.key_phrases}")
else:
    print(f"Errors: {result.parsing_errors}")

---

## Step 3: Nested Models

Structured output supports nested Pydantic models. The LLM generates
nested JSON and FFAI validates the full structure.

In [ ]:
class Address(BaseModel):
    city: str
    country: str

class CompanyInfo(BaseModel):
    name: str
    founded: int
    headquarters: Address
    products: list[str]

result = ffai.generate_response(
    prompt="Tell me about the company that makes the iPhone.",
    prompt_name="company_info",
    response_model=CompanyInfo,
)

if result.parsed:
    c = result.parsed
    print(f"Company:    {c.name}")
    print(f"Founded:    {c.founded}")
    print(f"HQ:         {c.headquarters.city}, {c.headquarters.country}")
    print(f"Products:   {c.products}")
else:
    print(f"Errors: {result.parsing_errors}")

---

## Step 4: Validation with Constraints

Pydantic `Field` constraints (like `ge`, `le`, `min_length`) are enforced.
If the LLM returns an out-of-range value, FFAI retries with error feedback.

In [ ]:
class RiskAssessment(BaseModel):
    risk_score: int = Field(ge=0, le=100, description="Risk score 0-100")
    risk_level: str = Field(description="One of: low, medium, high, critical")
    factors: list[str] = Field(min_length=1, description="At least one risk factor")
    recommendation: str

result = ffai.generate_response(
    prompt="Assess the risk of deploying an untested ML model to production on a Friday afternoon.",
    prompt_name="risk_1",
    response_model=RiskAssessment,
)

if result.parsed:
    r = result.parsed
    print(f"Score:           {r.risk_score}/100")
    print(f"Level:           {r.risk_level}")
    print(f"Factors:         {r.factors}")
    print(f"Recommendation:  {r.recommendation}")
    print(f"Cost:            ${result.cost_usd:.6f}")
else:
    print(f"Errors: {result.parsing_errors}")

---

## Step 5: Combined with `{{name.response}}` Interpolation

Structured output works seamlessly with FFAI's named-prompt interpolation.
First prompt produces a response, second prompt references it.

In [ ]:
ffai.generate_response(
    prompt="Write a short paragraph about renewable energy trends in 2025.",
    prompt_name="energy_text",
)

class SummaryWithTopics(BaseModel):
    summary: str = Field(description="One-sentence summary")
    topics: list[str] = Field(description="Key topics mentioned")
    sentiment: str = Field(description="Overall tone: optimistic, neutral, pessimistic")

result = ffai.generate_response(
    prompt="Summarize and extract topics from this text: {{energy_text.response}}",
    prompt_name="energy_summary",
    response_model=SummaryWithTopics,
)

if result.parsed:
    s = result.parsed
    print(f"Summary:   {s.summary}")
    print(f"Topics:    {s.topics}")
    print(f"Sentiment: {s.sentiment}")
else:
    print(f"Errors: {result.parsing_errors}")

---

## Step 6: Review History with Structured Results

All structured output calls are recorded in FFAI's history alongside
normal calls. The `parsed` field is available on the `ResponseResult`
but the raw response text is stored in history as usual.

In [ ]:
df = ffai.history_to_dataframe()

cols = [c for c in ['prompt_name', 'status', 'model'] if c in df.columns]
if not df.is_empty():
    print(f"History: {df.shape[0]} entries")
    print()
    print(df.select(cols))

total_cost = sum(
    getattr(r, 'cost_usd', 0) or 0
    for entry in ffai.history
)
print(f"\nTotal history entries: {len(ffai.history)}")

---

## Summary

| Parameter | Purpose |
|-----------|---------|
| `response_model=MyModel` | Pydantic model to validate the response against |

| `ResponseResult` field | Meaning |
|-----------------------|--------|
| `.parsed` | Validated Pydantic model instance (or `None`) |
| `.parsing_errors` | List of validation error strings (or `None`) |
| `.response` | Raw response text (always available) |

**How it works:**

1. FFAI converts the Pydantic model to JSON Schema
2. Schema instructions are appended to system prompt
3. LLM generates a response
4. `json_repair` parses the response (fault-tolerant)
5. Pydantic validates against the model
6. On failure: error feedback is sent back, LLM retries (up to 3 attempts total)
7. On success: `result.parsed` contains the validated instance